## Simulation of the control unit of the air handling device

In [1]:
import sys
sys.path.append('C:/Users/Acc/AppData/Local/Programs/Python/Python311/Lib/site-packages')
from time import sleep
import random
from BAC0.core.devices.local.object import ObjectFactory
from bacpypes.object import ScheduleObject
from bacpypes.primitivedata import Real
from BAC0 import lite, connect
from BAC0.core.devices.local.models import (
    analog_input,
    analog_output,
    binary_output,
    binary_input,
)
import csv
import time
import ifaddr

## Defined features for the controller
analog input:
- **CO2Sensor**
- **Humidity**
- **Off Coil Temperature**
- **Static Pressure**
- **Supply Air Temperature**

binary_output:
- **On-Off Status**

In [3]:
# define device objects
def defining_objects(AHU):
    # start from fresh
    ObjectFactory.clear_objects()
    
    #content of the object is defined here anward
    
    BA_AHU_1_1 = analog_input(
    instance=800,
    name = "CO2Sensor",
    properties = {"units" : "partsPerMillion"},
    description = "CO2Sensor",
    presentValue = 800.0,
    )
    analog_input(
        instance=75,
        name="Humidity",
        properties={"units" : "percent"},
        description="Humidity",
        presentValue=75.0,
    )
    analog_input(
        instance=21,
        name = "OffcoilTemp",
        properties={"units" : "degreesCelsius"},
        description="OffcoilTemp",
        presentValue=21.0,
    )
    analog_input(
        instance=11,
        name = "StaticPressure",
        properties={"units" : "pascals"},
        description="StaticPressure",
        presentValue=11.0,
    )
    analog_input(
        instance=23,
        name = "SupplyAirTemp",
        properties={"units" : "degreesCelsius"},
        description="SupplyAirTemp",
        presentValue=23.0,
    )

    binary_output(
        instance=0,
        name = "OnOffStatus",
        description="OnOffStatus",
        presentValue=True,
    )

    return BA_AHU_1_1.add_objects_to_application(AHU)

Configuring the controller to connect to the network involves:
- **reading the network interface IP**
- **using it for controller simulation**
- **specifying the port for the back-net protocol**
- **defining an ID for the controller**

In [ ]:
adapters = ifaddr.get_adapters()
ip = None
for adapter in adapters:
    if adapter.nice_name == 'maineth':
        ip = adapter.ip_addresses[0] 
        break

AHU = connect(ip=ip, port='47808', deviceId='3330')
defining_objects(AHU)

In [5]:
start_date = time.struct_time((2012, 2, 28, 16, 29, 10, 0, 0, 0))
start_timestamp = time.mktime(start_date)

for i in range(int(start_timestamp - time.time())):
  time.sleep(1)

with open('E://dataset_for_simulation/BA_AHU_1_1.csv') as f:
    reader = csv.reader(f)
    rows = list(reader)

row_num = 1 

while True:
    if row_num >= len(rows):
        break
        
    CO2Sensor = float(rows[row_num][1])
    Humidity = float(rows[row_num][2])
    OffcoilTemp = float(rows[row_num][3])
    StaticPressure = float(rows[row_num][5])
    SupplyAirTemp = float(rows[row_num][6])
    OnOffStatus = float(rows[row_num][4])
    
    if OnOffStatus == 0:
      OnOffStatus = False
    elif OnOffStatus == 1:
      OnOffStatus = True
        
    AHU["CO2Sensor"].presentValue = CO2Sensor 
    AHU["Humidity"].presentValue = Humidity
    AHU["OffcoilTemp"].presentValue = OffcoilTemp
    AHU["StaticPressure"].presentValue = StaticPressure
    AHU["SupplyAirTemp"].presentValue = SupplyAirTemp
    AHU["OnOffStatus"].presentValue = OnOffStatus
        

    print("AHU CO2Sensor :", AHU["CO2Sensor"].presentValue) 
    print("AHU Humidity :", AHU["Humidity"].presentValue)   
    print("AHU OffcoilTemp :", AHU["OffcoilTemp"].presentValue) 
    print("AHU StaticPressure :", AHU["StaticPressure"].presentValue) 
    print("AHU OnOffStatus :", AHU["OnOffStatus"].presentValue) 

    row_num += 1
    sleep(5)

print("Reached end of data")